# Parameters

In [1]:
# run_id = '21d445b4-c6a4-4ca7-b850-05124c6d8219'
# run_id = '8c4fe25b-7365-49ff-8963-48e1459a427b'
# run_id = 'ec89014e-e814-41a7-863c-265bb9557f8b'

# testing
run_id = 'a7a32475-50ee-4df8-9fb4-f0f4e1b1936d'

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 5, Finished, Available, Finished, False)

# Setup

In [2]:
# NOTE: Believe upgrade to Pandas changed PyArrow settings (Feb 26 2025)
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 6, Finished, Available, Finished, False)

In [3]:
import glob
import gurobipy as gp
import hive_optimizer
import json
import logging
import os
import pandas as pd
import shutil
import traceback
from enum import StrEnum
from hive_optimizer import utils
from hive_optimizer.schemas import input_schema, output_schema
from pathlib import Path
from packaging.version import Version
from pyspark.sql.types import IntegerType, LongType, DoubleType, NumericType, StringType, StructType
import numpy as np

# Constants
class RunStatuses(StrEnum):
    SUCCESS = "Success"
    INFEASIBLE = "Infeasible"
    FAILED = "Failed"

# Logging - Remove Old Files
outprefix = f'/lakehouse/default/Files/Logs/Run_{run_id}_'
files = glob.glob(f"{outprefix}*")
for file in files:
    if file.startswith(outprefix):
        os.remove(file)

# Logging - Filter Default Spark Messages
pyspark_log = logging.getLogger('pyspark').setLevel(logging.ERROR)
py4j_logger = logging.getLogger("py4j").setLevel(logging.ERROR)

# Logging - Configure
formatter = logging.Formatter(fmt="%(asctime)s %(levelname)s: %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
utils.setup_logging(
    debug_logfile=Path(f"{outprefix}debug.log"),
    data_logfile=Path(f"{outprefix}data.log"),
    formatter=formatter
)
logging.getLogger().setLevel(logging.INFO)

# Logging - Setup for Notebook
logger = logging.getLogger('DE_NB_RunModel')

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 7, Finished, Available, Finished, False)

In [4]:
# Check gurobi active process
os.system("ps aux | grep gurobi")

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 8, Finished, Available, Finished, False)

trusted+    8527  0.0  0.0  10708  3132 ?        S    20:54   0:00 sh -c ps aux | grep gurobi
trusted+    8532  0.0  0.0   9828  2548 ?        S    20:54   0:00 grep gurobi


0

# Helper Functions

In [5]:
def get_datatype(column_info):
    '''
    Return a pandas datatype for a column based on the `ticdat` column_info object
    '''
    if not column_info.number_allowed:
        return 'str'
    elif column_info.must_be_int:
        return 'int'
    else:
        return 'float'

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 9, Finished, Available, Finished, False)

In [6]:
def get_schema(ticdat_schema):
    '''
    Return Dict{table_name: (column_name, column_datatype)} for all tables in `ticdat_schema`
    '''
    schema_info = ticdat_schema.schema(include_ancillary_info=True)

    schema = {}
    for table, (key_fields, data_fields) in schema_info['tables_fields'].items():
        if table == 'parameters':
            continue

        columns = key_fields + data_fields
        infos = schema_info['data_types'][table]
        schema[table] = [
            (column, get_datatype(infos[column])) 
            for column in columns
        ]

    return schema

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 10, Finished, Available, Finished, False)

In [7]:
def create_empty_dataframe(column_infos):
    '''
    Return an empty DF with column names and datatypes matching `column_infos`

    Args:
    - column_infos List[(column_name, datatype)]
    '''
    columns = [column for column, datatype in column_infos]
    df = pd.DataFrame(columns=columns)
    df = df.astype({column: dtype for column, dtype in column_infos})
    return df

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 11, Finished, Available, Finished, False)

In [8]:
def create_parameters_df(run_id):
    '''
    Return `ticdat` parameters DF with data from `input_parameters` for `run_id`
    '''

    renames={'ProhibitMoneySamePeriod':'EnableC10'}

    sql = f"""
        SELECT *
        FROM input_parameters
        WHERE RunID = '{run_id}'
    """

    drop = ['RunID']
    df = spark.sql(sql).toPandas().rename(columns=renames).drop(columns=drop)
    df = pd.melt(df, var_name='Name', value_name='Value')

    # keep only parameters that are defined on the schema.
    # This allows running older model versions that don't implement newer parameteres
    df = df[df['Name'].isin(input_schema.parameters)].reset_index(drop=True)

    return df

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 12, Finished, Available, Finished, False)

In [9]:
def create_pivoted_kpis(model_input, model_output):
    '''
    Return the KPIs output DF pivoted with KPI names as columns
    '''
    df = model_output.kpis.copy()
    df = df.pivot_table(columns='Name', values='Value', aggfunc='first').reset_index()

    table = spark.sql("SELECT * FROM output_kpis WHERE 1 = 0")
    columns = [field.name for field in table.schema if field.name in df.columns]
    df = df[columns].copy()

    ##
    # Handle FirstCrossoverPoint[Label]
    ##

    # This was changed: Now the model returns the label as the FirstCrossoverPoint
    df = df.rename(columns={'FirstCrossoverPoint':'FirstCrossoverPointPeriodLabel'})
    try:
        label = df.FirstCrossoverPointPeriodLabel.iloc[0]

        # PeriodLabel is a python datetime
        periods = model_input.time_periods.copy()
        periods['PeriodLabel'] = pd.to_datetime(periods.PeriodLabel).dt.strftime('%Y-%m-%d')

        df['FirstCrossoverPoint'] = periods[periods.PeriodLabel == label].PeriodID.iloc[0]
    except:
        df['FirstCrossoverPoint'] = np.nan

    # Avoid attempt to insert Float (NAN) into Int column
    if df['FirstCrossoverPoint'].isna().all():
        df.drop(columns=['FirstCrossoverPoint'], inplace=True)

    return df

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 13, Finished, Available, Finished, False)

In [10]:
def create_model_input(run_id, input_schema):
    '''
    Return `ticdat.dat` model input created from data in lakehouse for RunID
    '''
    dfs = {}
    prefix = 'input_'

    # Reference Tables - Without RunID
    reference_tables = ['portfolios']

    # Parse the ticdat schema
    schema = get_schema(input_schema)

    # For each table in ticdat schema...
    for table, column_infos in schema.items():
        # Add empty DF if not in lakehouse
        if not spark.catalog.tableExists(prefix + table):
            dfs[table] = create_empty_dataframe(column_infos)
        # Else add DF with data from lakehouse table for RunID
        else:
            columns = [column for column, datatype in column_infos]

            sql = f"""
                SELECT {','.join(columns)}
                FROM {prefix}{table}
            """

            if table not in reference_tables:
                sql += f" WHERE RunID = '{run_id}' "

            dfs[table] = spark.sql(sql).toPandas()

    # Add parameters
    dfs['parameters'] = create_parameters_df(run_id)
    
    # return dfs
    return input_schema.PanDat(**dfs)

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 14, Finished, Available, Finished, False)

In [11]:
def save_model_output(run_id, model_input, model_output):
    '''
    Insert data from each DF in `model_output` into table.  Creates table if does not exist.
    '''

    prefix = 'output_'
    for field in output_schema.all_tables:
        df = getattr(model_output, field).copy()

        if df.empty:
            continue

        if 'PortfolioID' in df.columns:
            df["PortfolioID"] = df["PortfolioID"].replace("", np.nan)
        if len(df):
            print(f"Processing table {field}")

            if field == 'kpis':
                # Pivot KPI names to columns
                df = create_pivoted_kpis(model_input, model_output)

            # Add RunID
            df.insert(0, 'RunID', run_id)
            df['RunID'] = df['RunID'].astype(str)

            # Convert Categorical Values to Strings
            for col in df.select_dtypes(include=['category']).columns:
                df[col] = df[col].astype(str).replace('nan', '')
            
            # Convert pandas.NA values to None
            if field == 'ledger':
                df['ReferencePeriodID'] = np.where(df['ReferencePeriodID'].isna(), None, df['ReferencePeriodID'])

            # Build Table & Temp View names
            table_name = f'{prefix}{field}'
            view_name = f'temp_{table_name}_run{run_id}'.replace('-', '_')

            # Scrub column names
            df.columns = df.columns.str.replace(r'[^a-zA-Z0-9_]', '', regex=True)

            # Get Table Schema
            schema = spark.table(table_name).schema
            # Create Dict of Table Column Defs
            schema_columns = {'DeployToRepeats' if c.name == 'DeploytoRepeats' else c.name: c for c in schema}
            # Get DF columns in Table
            columns = [c for c in df.columns if c in schema_columns]

            # Create Spark DF
            spark_df = spark.createDataFrame(df[columns])

            # Convert Datatypes per Schema
            for c in columns:
                field = schema_columns[c]
                if isinstance(field.dataType, IntegerType):
                    spark_df = spark_df.withColumn(field.name, spark_df[field.name].cast(IntegerType()))
                elif isinstance(field.dataType, LongType):
                    spark_df = spark_df.withColumn(field.name, spark_df[field.name].cast(LongType()))
                elif isinstance(field.dataType, DoubleType):
                    spark_df = spark_df.withColumn(field.name, spark_df[field.name].cast(DoubleType()))
                elif isinstance(field.dataType, StringType):
                    spark_df = spark_df.withColumn(field.name, spark_df[field.name].cast(StringType()))

            # Fill Numeric NA Values with Zero
            replacements = {}
            fill_values = {
                IntegerType(): 0,
                LongType(): 0,   # Or 0L in some cases
                DoubleType(): 0.0
            }

            for field in spark_df.schema.fields:
                if isinstance(field.dataType, NumericType):
                    # Use type-specific defaults
                    replacements[field.name] = fill_values.get(type(field.dataType), 0)

            spark_df = spark_df.fillna(replacements)

            # Create a Temp View
            spark_df.createOrReplaceTempView(view_name)

            # Create a SELECT clause
            columns_clause = ",".join(c for c in df.columns)

            if not spark.catalog.tableExists(table_name):
                # If table does not exist, create it from view
                sql = f"""
                CREATE TABLE {table_name} AS 
                SELECT {columns_clause}
                FROM {view_name}
                """
                # print("Creating table:", table_name)
            else:
                sql = f"""
                DELETE FROM {table_name}
                WHERE RunID = '{run_id}'
                """
                # print(sql)
                spark.sql(sql)

                # Else insert into it from view
                sql = f"""
                INSERT INTO {table_name}
                ({columns_clause})
                SELECT {columns_clause}
                FROM {view_name}
                """
                print("Inserting into table:", table_name)
                
            logger.debug("****************")
            logger.debug(table_name)
            logger.debug("****************")
            logger.debug(sql)
            logger.debug("****************")
            spark.sql(sql)

    # Fill output_ledger.PeriodLabel
    # TEMP: Remove bkp
    sql = f"""
    MERGE INTO output_ledger AS target
    USING input_time_periods AS source
    ON target.RunID = source.RunID
    AND target.PeriodID = source.PeriodID
    WHEN MATCHED AND target.RunID = '{run_id}' THEN
    UPDATE SET target.PeriodLabel = source.PeriodLabel;
    """
    spark.sql(sql)

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 15, Finished, Available, Finished, False)

# Main

In [12]:
def main(run_id):
    status = RunStatuses.FAILED
    error_message = ''

    try:
        # Store Model Version
        model_version = hive_optimizer.__version__
        spark.sql(f"UPDATE runs SET ModelVersion = '{model_version}' WHERE RunID = '{run_id}'")
        
        # Run Model
        model_input = create_model_input(run_id, input_schema)
        model_output = hive_optimizer.optimize(model_input)

        # Save Output
        save_model_output(run_id, model_input, model_output)

        # Update Run to Success
        is_infeasible = all(getattr(model_output, field).empty for field in output_schema.all_tables if field != 'kpis')
        status = RunStatuses.INFEASIBLE if is_infeasible else RunStatuses.SUCCESS
    except Exception as e:
        # Log Error
        error_message = traceback.format_exc()
        logging.error(f"An error occurred: {e}")
        logging.error(error_message)
        
        
        # Update Run to Failed
        status = RunStatuses.FAILED
        error_message = str(e).replace("\n", " ").replace("'", "''")
    finally:
        spark.sql(f"UPDATE runs SET Status = '{status.value}', ErrorMessage = '{error_message}' WHERE RunID = '{run_id}'")

    return {'run_id': run_id, 'status': status.value, 'error_message': json.dumps(error_message)}

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 16, Finished, Available, Finished, False)

In [13]:
# point to gurobi license
os.environ["GRB_LICENSE_FILE"] = '/lakehouse/default/Files/gurobi.lic'

# run main engine
version = Version(hive_optimizer.__version__)
if version < Version("1.11.1") and version != Version("1.9.5"):
    # manage the gurobi resources explicitly
    with gp.Env() as env:
        with gp.Model(env=env) as model:
            result = main(run_id)
else:
    # hive_optimizer package handles the resources under the hood
    result = main(run_id)

mssparkutils.notebook.exit(str(result))

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, 17, Finished, Available, Finished, False)

2026-06-01 20:56:23 INFO: Welcome to the Hive Optimizer engine!
2026-06-01 20:56:36 INFO: Preparing the optimization data (from processed tables)
2026-06-01 20:56:36 INFO: Scaling data
2026-06-01 20:56:36 INFO: Running data integrity checks
2026-06-01 20:56:58 WARNING: Finished populating the optimization data with warnings. See data log for details
2026-06-01 20:56:58 INFO: Creating the optimization model
2026-06-01 21:03:43 INFO: Set solver parameter 'LogToConsole'=0
2026-06-01 21:03:43 INFO: Set solver parameter 'LogFile'='/lakehouse/default/Files/Logs/Run_a7a32475-50ee-4df8-9fb4-f0f4e1b1936d_debug.log'
2026-06-01 21:03:43 INFO: Set solver parameter 'PreDual'=1
2026-06-01 21:03:43 INFO: Set solver parameter 'Method'=0
2026-06-01 21:03:43 INFO: Optimizing single objective 'Maximize Bank Balance'
2026-06-01 21:05:40 INFO: Optimization status MainModel: OPTIMAL
2026-06-01 21:05:46 INFO: Preparing output tables
2026-06-01 21:12:18 INFO: Optimization engine completed!
2026-06-01 21:12:18

Processing table financial
Inserting into table: output_financial
Processing table ledger
Inserting into table: output_ledger
Processing table kpis
Inserting into table: output_kpis
Processing table ledger_pivot
Inserting into table: output_ledger_pivot
Processing table balances
Inserting into table: output_balances
ExitValue: {'run_id': 'a7a32475-50ee-4df8-9fb4-f0f4e1b1936d', 'status': 'Success', 'error_message': '""'}

# Troubleshooting

In [ ]:
message = result['error_message']
lines = message.split('\\n')
for line in lines:
    print(line.strip())

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, -1, Cancelled, , Cancelled, True)

## Exporting data

In [ ]:
import pandas as pd
import base64
from io import BytesIO, StringIO
from IPython.display import HTML
from datetime import date
import zipfile


def export_csv(data: dict[str, pd.DataFrame]):
    # Create a zip file in memory
    zip_buffer = BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zip_file:
        for table_name, df in data.items():
            # Create CSV content for each table
            csv_buffer = StringIO()
            df.to_csv(csv_buffer, index=False)
            csv_content = csv_buffer.getvalue()
            
            # Add CSV file to the zip archive
            zip_file.writestr(f'{table_name}.csv', csv_content)
            
    # Encode as base64
    b64 = base64.b64encode(zip_buffer.getvalue()).decode()

    # Generate a download link
    return HTML(f'<a download="exported_data.zip" href="data:application/zip;base64,{b64}">Download CSV Zip</a>')


def export_excel(data: dict[str, pd.DataFrame]):
    # Create an Excel file in memory
    output = BytesIO()
    with pd.ExcelWriter(output, engine="xlsxwriter") as writer:
        for table_name, df in data.items():
            df.to_excel(writer, sheet_name=table_name, index=False)
        writer.close()

    # Encode as base64
    b64 = base64.b64encode(output.getvalue()).decode()

    # Generate a download link
    return HTML(f'<a download="exported_data.xlsx" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}">Download Excel</a>')

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, -1, Cancelled, , Cancelled, True)

In [ ]:
# input
# model_input = create_model_input(run_id, input_schema)
# data = {table_name: getattr(model_input, table_name) for table_name in input_schema.all_tables}

# output
# data = {}
# for table_name in output_schema.all_tables:
#     df = spark.sql(f"SELECT * FROM output_{table_name} WHERE RunID = '{run_id}'").toPandas()
#     df = df.drop(columns=['RunID'])
#     if table_name == 'kpis':
#         df = pd.melt(df, var_name='Name', value_name='Value')
#     data[table_name] = df

# export
# download_link = export_csv(data)
# download_link = export_excel(data)
# display(download_link)

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, -1, Cancelled, , Cancelled, True)

In [ ]:
df = spark.sql("SELECT * FROM DE_LH_HiveGlobalOptimizer.committed_roll")

StatementMeta(, c1e8a772-dff2-4f17-9884-37d3a6850015, -1, Cancelled, , Cancelled, True)